In [ ]:
"""
Author: Isabel G.
Last edited: 7/2/2025

This file was used to derive a new formulation of state purity in terms of tr(rho^2) and chi so that the
purity is the same across the full range of chi values. If the purity calculation is called into question
in the future, this file can be used as a reference.
"""

In [ ]:
from sympy import *
from sympy.parsing.mathematica import parse_mathematica
import numpy as np
import states_and_witnesses as sw
import operations as op

In [3]:
# Define kets in vector form 
H = op.ket([1,0])
V = op.ket([0,1])
R = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (1j)])
L = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (-1j)])
D = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (1)])
A = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (-1)])

In [ ]:
# Create symbolic variables
p = Symbol("p", positive=True) # purity
q = Symbol("q", positive=True) # trace of rho squared
chi_var = Symbol("chi_var") # symbolic chi (not to be confused with chi, which we plug in numerically)

In [5]:
def phi1(alpha, alpha_perp, chi, phase=0):
    """
    phi1 is the target state we are creating with probability p.
    """
    return cos(chi/2) * np.kron(H, alpha) + exp(1j * phase) * sin(chi/2) * np.kron(V, alpha_perp)

def phi2(alpha):
    """
    phi2 is the state |H>|alpha> that we create with probability (1-p)cos^2(chi/2).
    """
    return np.kron(H, alpha)

def phi3(alpha_perp):
    """
    phi3 is the state |V>|alpha_perp> that we create with probability (1-p)sin^2(chi/2).
    """
    return np.kron(V, alpha_perp)

In [6]:
def rho(p, alpha, alpha_perp, chi, phase=0):
    """
    Constructs the density matrix based on the probabilities of each state created.
    """
    target_rho = p * (phi1(alpha, alpha_perp, chi, phase) * op.adjoint(phi1(alpha, alpha_perp, chi, phase)))
    phi2_rho = (1-p)*(cos(chi/2))**2 * np.outer(np.kron(H, alpha), op.adjoint(np.kron(H, alpha)))
    phi3_rho = (1-p)*(sin(chi/2))**2 * np.outer(np.kron(V, alpha_perp), op.adjoint(np.kron(V, alpha_perp)))
    return target_rho + phi2_rho + phi3_rho

In [75]:
# The following 3 testing blocks create rhos for HH_VV, HD_VA, and HR_VL
rho1 = rho(p, H, V, pi/2)
print(rho1)
rho1_sq = rho1 @ rho1
print("Trace of rho squared:", np.trace(rho1_sq))

[[0.500000000000000 0 0 0.5*p]
 [0 0 0 0]
 [0 0 0 0]
 [0.5*p 0 0 0.500000000000000]]
Trace of rho squared: 0.5*p**2 + 0.5


In [76]:
rho2 = rho(p, D, A, pi/2)
print(rho2)
rho2_sq = rho2 @ rho2
print("Trace of rho squared:", np.trace(rho2_sq))

[[0.250000000000000 0.250000000000000 0.25*p -0.25*p]
 [0.250000000000000 0.250000000000000 0.25*p -0.25*p]
 [0.25*p 0.25*p 0.250000000000000 -0.250000000000000]
 [-0.25*p -0.25*p -0.250000000000000 0.250000000000000]]
Trace of rho squared: 0.5*p**2 + 0.5


In [78]:
rho3 = rho(p, R, L, pi/2)
print(rho3)
rho3_sq = rho3 @ rho3
print("Trace of rho squared:", np.trace(rho3_sq))

[[0.250000000000000 -0.25*I*p - 0.5*I*(1/2 - p/2) 0.25*p 0.25*I*p]
 [0.25*I*p + 0.5*I*(1/2 - p/2) 0.250000000000000 0.25*I*p -0.25*p]
 [0.25*p -0.25*I*p 0.250000000000000 0.25*I*p + 0.5*I*(1/2 - p/2)]
 [-0.25*I*p -0.25*p -0.25*I*p - 0.5*I*(1/2 - p/2) 0.250000000000000]]
Trace of rho squared: 0.5*p**2 + 4*(-0.25*I*p - 0.5*I*(1/2 - p/2))*(0.25*I*p + 0.5*I*(1/2 - p/2)) + 0.25


In [82]:
# Solve for purity in terms of the trace of rho^2 for the chi = pi/2 case
equation = Eq(np.trace(rho1_sq), q)
print(solveset(equation, p))

{-1.4142135623731*sqrt(1.0*q - 0.5), 1.4142135623731*sqrt(1.0*q - 0.5)}


In [83]:
# chi = 0.001 case, shows that purity is dependent on chi
rho5 = rho(p, H, V, 0.001)
rho5_sq = rho5 @ rho5
print(np.trace(rho5_sq))
equation = Eq(np.trace(rho5_sq), q)
print(solveset(equation, p))

4.99999833333356e-7*p**2 + 0.999999500000167
{-1414.21379807538*sqrt(1.0*q - 0.999999500000167), 1414.21379807538*sqrt(1.0*q - 0.999999500000167)}


In [ ]:
# Calculate purity for full range of chi values
chis = np.linspace(0.001, np.pi/2, 6)
for chi in chis:
    this_rho = rho(p, H, V, chi)
    tr_rho_sq = np.trace(this_rho @ this_rho)
    print(f"\nchi: {np.rad2deg(chi)}, trace(rho^2): {tr_rho_sq}")
    equation = Eq(tr_rho_sq, q)
    print(f"chi: {np.rad2deg(chi)}, p: {solveset(equation, p)}")


chi: 0.057295779513082325, trace(rho^2): 4.99999833333356e-7*p**2 + 0.999999500000167
chi: 0.057295779513082325, p: {-1414.21379807538*sqrt(1.0*q - 0.999999500000167), 1414.21379807538*sqrt(1.0*q - 0.999999500000167)}

chi: 18.045836623610466, trace(rho^2): 0.0479811242922478*p**2 + 0.952018875707752
chi: 18.045836623610466, p: {-4.56525236298862*sqrt(1.0*q - 0.952018875707752), 4.56525236298862*sqrt(1.0*q - 0.952018875707752)}

chi: 36.03437746770785, trace(rho^2): 0.173031123915728*p**2 + 0.826968876084272
chi: 36.03437746770785, p: {-2.40401894394615*sqrt(1.0*q - 0.826968876084272), 2.40401894394615*sqrt(1.0*q - 0.826968876084272)}

chi: 54.022918311805235, trace(rho^2): 0.327444435155348*p**2 + 0.672555564844652
chi: 54.022918311805235, p: {-1.74755636800476*sqrt(1.0*q - 0.672555564844652), 1.74755636800476*sqrt(1.0*q - 0.672555564844652)}

chi: 72.01145915590261, trace(rho^2): 0.452313010937059*p**2 + 0.547686989062941
chi: 72.01145915590261, p: {-1.48689554324964*sqrt(1.0*q - 0.

In [ ]:
# Build rho without plugging in chi and solve for p
test_rho = rho(p, H, V, chi_var)
tr_rho_sq = np.trace(test_rho @ test_rho)
equation = Eq(tr_rho_sq, q)
print(f"p: {solveset(equation, p)}")

p: {-sqrt(-(-1.0*q + 0.5*(cos(chi_var) - 1)**2 + 1.0*cos(chi_var))*(8.0*sin(chi_var/2)**4 + 8.0*sin(chi_var/2)**2*sin(conjugate(chi_var)/2)**2 + 8.0*sin(chi_var/2)**2*cos(chi_var/2 + conjugate(chi_var)/2) - 12.0*sin(chi_var/2)**2 + 2.0*sin(chi_var)*sin(conjugate(chi_var)) - 4.0*sin(conjugate(chi_var)/2)**2 - 8.0*cos(chi_var/2)*cos(conjugate(chi_var)/2) + 8.0) + 0.25*(cos(2*chi_var) - 3*cos(chi_var/2 - conjugate(chi_var)/2) - cos(3*chi_var/2 + conjugate(chi_var)/2) + 3)**2)/(2.0*sin(chi_var/2)**4 - 4.0*sin(chi_var/2)**3*sin(conjugate(chi_var)/2) + 2.0*sin(chi_var/2)**2*sin(conjugate(chi_var)/2)**2 + 4.0*sin(chi_var/2)*sin(conjugate(chi_var)/2)*cos(chi_var/2)*cos(conjugate(chi_var)/2) + 2.0*cos(chi_var/2)**4 - 4.0*cos(chi_var/2)**3*cos(conjugate(chi_var)/2) + 2.0*cos(chi_var/2)**2*cos(conjugate(chi_var)/2)**2) + (-0.166666666666667*sin(chi_var)**2 - 0.25*cos(chi_var/2 - conjugate(chi_var)/2) - 0.0833333333333333*cos(3*chi_var/2 + conjugate(chi_var)/2) + 0.333333333333333)/(0.666666666666

In [84]:
def p_eval_long(q, chi):
    """
    Purity calculation based on Sympy's solution.
    """
    sol = sqrt((q - 0.5*cos(chi)**2 - 0.5)*(16.0*sin(chi/2)**4 +
            8.0*sin(chi/2)**2*cos(chi) - 16.0*sin(chi/2)**2 + 2.0*sin(chi)**2 -
            8.0*cos(chi/2)*cos(chi/2) + 8.0))/(4.0*sin(chi/2)**2*cos(chi/2)**2) + (-(1/6)*sin(chi)**2 -
            0.25 - (1/12)*cos(2*chi) + (1/3))/((4/3)*sin(chi/2)**4 +
            (2/3)*sin(chi/2)**2*cos(chi) - sin(chi/2)**2 -
            (1/6)*sin(chi)**2 - (2/3)*cos(chi/2)**2 + (2/3))
    return sol
def p_eval(q, chi):
    """
    Purity calculation based on Isabel's math done by hand.
    """
    return sqrt(2*(q-1) / sin(chi)**2 + 1)

In [87]:
# Plug in purity = 0.95 and verify that the program returns that for all chis
for chi in chis:
    this_rho = rho(0.95, H, V, chi)
    tr_rho_sq = np.trace(this_rho @ this_rho)
    print(f"chi: {np.rad2deg(chi)}, p: {p_eval(tr_rho_sq, chi)}")

chi: 0.057295779513082325, p: 0.950000000152119
chi: 18.045836623610466, p: 0.950000000000000
chi: 36.03437746770785, p: 0.950000000000000
chi: 54.022918311805235, p: 0.950000000000000
chi: 72.01145915590261, p: 0.950000000000000
chi: 90.0, p: 0.950000000000000
